# Qwen2.5-1.5B DeepReasoning ablation

This executed notebook is the presentation artifact. It shows the training/evaluation code path, but loads saved cloud artifacts so the notebook is fast and reproducible during class.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, Image, display

ART = Path('artifacts_qwen15b')
df = pd.read_csv(ART / 'qwen15b_screen_summary.csv')
confirm = json.loads((ART / 'confirm_qwen15_len1024_r8_a16_lr5e5_e3_es_greedy100' / 'evaluation_summary.json').read_text())
report_md = (ART / 'QWEN15B_PRESENTATION_REPORT.md').read_text()

def pct(x):
    return f'{100*x:.1f}%'

print('Loaded artifacts from', ART)
print('Screened runs:', len(df))
print('Confirmation examples:', confirm['n_examples'])

## 1. Research question and conclusion

We switched from Qwen2.5-3B to Qwen2.5-1.5B because the 3B model was strong enough that the dataset was less revealing. The adapter learns the required reasoning format, but does not improve loose final-answer accuracy over the base model.

In [ ]:
display(Markdown(report_md))

## 2. Training configuration we actually used

The next cells show the training setup. `RUN_FULL_TRAINING` is deliberately `False`: the heavy training already ran on the RTX PRO 6000 and the notebook loads those saved artifacts.

In [ ]:
# This cell documents the actual training configuration.
# It is not executed during presentation; the executed results are loaded from artifacts.
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
    DATA_PATH = 'data/sft_reasoning_2k.jsonl'
    ARTIFACT_ROOT = 'artifacts_qwen15b'

    MAX_LEN = 1024
    LORA_R = 8
    LORA_ALPHA = 16
    LEARNING_RATE = 5e-5
    NUM_EPOCHS = 3

    TRAIN_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 4
    EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    EVAL_BATCH_SIZE = 8
    GRADIENT_CHECKPOINTING = False

    EVAL_PROBLEMS = 30
    MAX_NEW_TOKENS = 768
    RUN_SELF_CONSISTENCY = False

## 3. Core QLoRA training code

This is the implementation: 4-bit NF4 loading, LoRA target modules, Trainer settings, early stopping, and adapter saving.

In [ ]:
# Core QLoRA training cell used by the experiment.
# Kept behind RUN_FULL_TRAINING so this presentation notebook opens quickly.
if RUN_FULL_TRAINING:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback, Trainer, TrainingArguments
    import torch

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization,
        device_map={'': 0},
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
    )
    model = get_peft_model(model, lora_config)

    args = TrainingArguments(
        output_dir=f'{ARTIFACT_ROOT}/{RUN_NAME}/checkpoints',
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='paged_adamw_8bit',
        bf16=True,
        tf32=True,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        logging_steps=1,
        eval_strategy='steps',
        eval_steps=20,
        save_strategy='steps',
        save_steps=20,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True,
                                             label_pad_token_id=-100,
                                             pad_to_multiple_of=8),
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3,
                                         early_stopping_threshold=5e-4)],
    )
    trainer.train()
    trainer.evaluate()
    trainer.save_model(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

## 4. Evaluation code and why there are two metrics

Strict accuracy asks whether the model obeyed the required format. Loose accuracy asks whether the final numeric answer can be recovered anywhere in the trace.

In [ ]:
# Evaluation logic used for both the 30-question screen and 100-question confirmation.
# Strict accuracy requires the model to place the final answer inside <answer> tags.
# Loose accuracy extracts the last number from the whole trace, even if tags are missing.
import re

SYSTEM_INSTRUCTION = """You are a meticulous reasoning tutor.
For every problem, answer using EXACTLY these tags in order:
<thinking>
Reason step by step. Show every intermediate calculation.
</thinking>
<reflection>
Re-check your reasoning. Look for arithmetic slips or wrong assumptions.
</reflection>
<answer>
Give only the final answer.
</answer>"""

def extract_tag(text: str, tag: str) -> str:
    match = re.search(fr'<{tag}>(.*?)</{tag}>', text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ''

def normalize_number(text: str) -> str:
    numbers = re.findall(r'-?\d+(?:\.\d+)?', str(text).replace(',', '').replace('$', ''))
    if not numbers:
        return ''
    value = float(numbers[-1])
    return str(int(value)) if value.is_integer() else str(value)

def score_trace(trace: str, gold: str) -> dict:
    thinking = extract_tag(trace, 'thinking')
    reflection = extract_tag(trace, 'reflection')
    answer = extract_tag(trace, 'answer')
    strict_prediction = normalize_number(answer)
    loose_prediction = strict_prediction or normalize_number(trace)
    return {
        'valid_format': bool(thinking and reflection and answer),
        'has_reflection': bool(reflection),
        'strict_correct': strict_prediction == gold,
        'loose_correct': loose_prediction == gold,
    }

## 5. Hyperparameter grid and saved training artifacts

This table is loaded from artifacts, not recomputed. It is the evidence used to select the leader.

In [ ]:
cols = ['run', 'rank', 'alpha', 'lr', 'train_batch', 'grad_accum', 'effective_batch', 'eval_loss', 'strict_accuracy_30', 'loose_accuracy_30', 'format_rate_30', 'train_minutes', 'max_gpu_gib']
display(df[cols])

## 6. Charts: screen accuracy and loss/accuracy relationship

The r=16 adapter had the best validation loss but worse strict accuracy. That is why the final choice uses held-out generation behavior, not validation loss alone.

In [ ]:
display(Image(filename=str(ART / 'presentation_plots' / 'screen_accuracy.png')))
display(Image(filename=str(ART / 'presentation_plots' / 'loss_vs_accuracy.png')))

## 7. Training stability and runtime choices

Batch 32 and batch 16 OOMed; batch 8 with grad accumulation 4 was stable and preserved effective batch size 32.

In [ ]:
display(Image(filename=str(ART / 'presentation_plots' / 'validation_curves.png')))
for oom_dir in sorted(ART.glob('oom_*')):
    record = oom_dir / 'OOM_RECORD.md'
    if record.exists():
        print(f'--- {oom_dir.name} ---')
        print(record.read_text())

## 8. 100-question confirmation

After the 30-question screen, only the leader was confirmed on 100 questions without retraining.

In [ ]:
display(Image(filename=str(ART / 'presentation_plots' / 'confirm_accuracy.png')))
display(Image(filename=str(ART / 'presentation_plots' / 'confirm_format_rate.png')))
print(json.dumps(confirm, indent=2))

## 9. Final takeaway

Use `qwen15_len1024_r8_a16_lr5e5_e3_es` if structured reasoning traces are required. If the only objective is loose GSM8K numeric answer accuracy, the base 1.5B model is still slightly better. The experiment demonstrates a format-following gain with a raw-answer tradeoff.